# Clasificación binaria

Las técnicas de **aprendizaje supervisado** entrenan un modelo sobre un conjunto de *features* para predecir un *label*:

> f([x1, x2, x3, ...]) = y

La **clasificación** predice a qué clase pertenece una observación, calculando la probabilidad de cada clase posible. Su forma más simple es la **binaria**: el label es 0 o 1 ("verdadero"/"falso", "aprobado"/"rechazado", etc.).

Aquí entrenamos un clasificador binario para predecir si un paciente debería hacerse una prueba de diabetes según sus datos médicos.

> **Cita**: el dataset de diabetes se basa en datos recopilados originalmente por el *National Institute of Diabetes and Digestive and Kidney Diseases*.

In [ ]:
import pandas as pd

# Cargar el dataset
diabetes = pd.read_csv('../datasets/diabetes.csv')
diabetes.head()

Son datos de diagnóstico de pacientes a los que se les hizo la prueba de diabetes. La última columna, **Diabetic**, vale `0` si dio negativo y `1` si dio positivo — ese es el label que queremos predecir. El resto (Pregnancies, PlasmaGlucose, DiastolicBloodPressure, etc.) son las features.

Separemos features (`X`) del label (`y`):

In [ ]:
features = ['Pregnancies','PlasmaGlucose','DiastolicBloodPressure','TricepsThickness',
            'SerumInsulin','BMI','DiabetesPedigree','Age']
label = 'Diabetic'

X, y = diabetes[features].values, diabetes[label].values

for n in range(4):
    print(f'Paciente {n+1}\n  Features: {list(X[n])}\n  Label: {y[n]}')

## Comparar la distribución de cada feature según el label

Si una feature se distribuye de forma distinta para pacientes diabéticos y no diabéticos, es señal de que puede ayudar a predecir.

In [ ]:
from matplotlib import pyplot as plt

for col in features:
    diabetes.boxplot(column=col, by='Diabetic', figsize=(6,6))
    plt.title(col)
plt.show()

En varias features se nota la diferencia. **Pregnancies** y **Age** en particular muestran distribuciones marcadamente distintas entre pacientes diabéticos y no diabéticos — son buenas candidatas a ser predictivas.

## Dividir los datos

Tenemos los labels conocidos, así que podríamos entrenar con todo. Pero entonces, ¿cómo sabríamos si el modelo predice bien con datos que **no ha visto**?

La solución es reservar una parte: entrenar con el 70% y evaluar con el 30% restante, comparando las predicciones contra los labels que ya conocemos.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

print(f'Casos de entrenamiento: {X_train.shape[0]}')
print(f'Casos de prueba:        {X_test.shape[0]}')

## Entrenar el modelo

Usamos **regresión logística** que, pese al nombre, es un algoritmo bien establecido de *clasificación*.

Además de las features y labels, hay que fijar un parámetro de **regularización**. Sirve para contrarrestar sesgos de la muestra y ayudar a que el modelo generalice, evitando que se sobreajuste a los datos de entrenamiento.

> En scikit-learn, `LogisticRegression` recibe `C`, que es la **inversa** de la tasa de regularización. Por eso se pasa `C=1/reg`: a menor `C`, mayor regularización.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Tasa de regularización
reg = 0.01

model = LogisticRegression(C=1/reg, solver='liblinear').fit(X_train, y_train)
print(model)

## Evaluar

Predecimos sobre los datos reservados y comparamos con los labels reales.

In [ ]:
predictions = model.predict(X_test)
print('Labels predichos:', predictions[:25])
print('Labels reales   :', y_test[:25])

Los arrays son demasiado largos para compararlos a ojo — y aunque los imprimiéramos enteros, no sería una forma sensata de evaluar el modelo.

Para eso están las **métricas**. La primera y más obvia es la **exactitud** (*accuracy*): ¿qué proporción de los labels predijo correctamente?

In [ ]:
from sklearn.metrics import accuracy_score

print('Exactitud:', accuracy_score(y_test, predictions))

La exactitud se devuelve como decimal: 1.0 sería acertar el 100% de las predicciones.

## Resumen

Preparamos los datos dividiéndolos en entrenamiento y prueba, y aplicamos regresión logística para asignar labels binarios. El modelo predice si un paciente tiene diabetes con una exactitud que *parece* razonable.

Pero, ¿es suficiente? **No.** En el cuaderno `02-metricas-clasificacion.ipynb` vemos por qué la exactitud sola puede ser muy engañosa, y qué métricas usar en su lugar.